<style>table { margin-left: 0 !important; } td, th { text-align: left !important; }</style>

# SageAgent V4.8.0

AI coding assistant for SageMaker notebooks. 25+ tools, 16 security layers, prompt caching, sub-agent coordination, 10 skills, Runnable-grade review/verification, local-git regression protection. **v4.8.0**.

**Setup:** Run cells 1-3 in order. Cell 1 installs packages (once). Cell 2 shows config widgets. Cell 3 launches the agent.

**Core files:** `sagemaker_agent.py` (~9,950 lines — the entire agent) + this notebook + `memory.md` (auto-populated) + `skills/` (10 skills).

**Docs:** See `USER_GUIDE.md` for full documentation + `changelogs/CHANGELOG_v4.8.0.md` for release notes. Cell 4 below has a quick reference.

## What is new in v4.8.0

**Major UX + harness engineering release** based on PS_Deep Runnable architecture research (2,010-file deep dive).

| Change | Type | Details |
|--------|------|---------|
| Chat window resizable | UX | Default 500px, drag bottom edge or use Chat Height slider (200-1200px) |
| Prefer chat answers | [CRITICAL] | Agent answers in chat, NOT by creating .md files. Only creates files when explicitly asked. |
| CSV/Excel validation | [CRITICAL] | Agent validates row counts, checks joins for duplicates/Cartesian products, flags inflated data. |
| wget/bash allowed | Security | `wget` and `bash` scripts now work. Only pipe-to-shell (`wget ... \| bash`) still blocked. |
| Budget display-only | UX | Budget shows cost but never stops the agent. Editable text input (0-999$). |
| Skills user-controlled | UX | All skills require explicit `/command` — no auto-triggering on generic keywords. |
| Post-compact cleanup | Harness | FILE_CACHE cleared after compaction so agent re-reads files correctly. |
| Per-turn reminders | Harness | Critical behavioral rules re-injected each LLM turn to prevent drift. |
| Cache miss warning | Harness | Shows cost spike warning after compaction breaks prompt cache. |
| coding-standards removed | Cleanup | Overlapped with `/simplify`. Use `/simplify` instead. |

## How It Works (30-second version)

1. You type a message → sent to Claude via AWS Bedrock
2. Claude picks the right tool (read file, run bash, edit code, etc.) → executes it locally
3. Result sent back to Claude → repeats until done (up to 60 turns)
4. **Skills** = instruction files activated via `/command` (review, verify, simplify, security-review, batch, report, design, clara-review, powerbi x2)
5. **Memory** = persistent facts the agent learns about you across sessions (`memory.md`)
6. **Security** = 16 layers validate every tool call before execution

## Quick Start

```
"Read main.py and explain what it does"       ← just ask, no skill needed
"Fix the bug in parser.py line 42"            ← agent reads, fixes, verifies
"Create a bar chart from sales.csv"           ← uses create_chart tool
/verify                                       ← adversarial testing after edits
/done quick                                   ← simplify + verify gate (when ready to ship)
/cost                                         ← check token usage and spend
```

In [ ]:
# Install dependencies (run once)
!pip install -q boto3 ipywidgets Pillow python-docx pandas openpyxl

In [ ]:
# ============================================================
# CONFIGURATION
# ============================================================
# Models are imported from sagemaker_agent.py (single source of truth)

import ipywidgets as widgets
from IPython.display import display, HTML
from sagemaker_agent import BEDROCK_MODELS

# Convert BEDROCK_MODELS list-of-tuples to dict for config cell
AVAILABLE_MODELS = dict(BEDROCK_MODELS)

# Temperature options
TEMPERATURE_OPTIONS = {
    "0.0 - Deterministic": 0.0,
    "0.3 - Low creativity": 0.3,
    "0.5 - Balanced": 0.5,
    "0.7 - High creativity": 0.7,
    "1.0 - Maximum creativity": 1.0,
}

# Thinking budget options
THINKING_BUDGET_OPTIONS = {
    "1024 - Minimal": 1024,
    "2048 - Light": 2048,
    "4096 - Standard": 4096,
    "8192 - Extended": 8192,
    "16000 - Maximum": 16000,
}

# Region - Sydney (ap-southeast-2)
REGION = "ap-southeast-2"

# Create configuration widgets
display(HTML("<h3>Agent Configuration</h3>"))

# Use first model as default (matches BEDROCK_MODELS order)
default_model_name = list(AVAILABLE_MODELS.keys())[0]

model_dropdown = widgets.Dropdown(
    options=list(AVAILABLE_MODELS.keys()),
    value=default_model_name,
    description='Model:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='450px')
)

temperature_dropdown = widgets.Dropdown(
    options=list(TEMPERATURE_OPTIONS.keys()),
    value="0.0 - Deterministic",
    description='Temperature:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='350px')
)

thinking_checkbox = widgets.Checkbox(
    value=False,
    description='Enable Extended Thinking (slower, uses more tokens)',
    indent=False,
    style={'description_width': 'auto'}
)

thinking_budget_dropdown = widgets.Dropdown(
    options=list(THINKING_BUDGET_OPTIONS.keys()),
    value="4096 - Standard",
    description='Thinking Budget:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='350px')
)

max_turns_slider = widgets.IntSlider(
    value=60,
    min=5,
    max=100,
    step=5,
    description='Max Turns:',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

workspace_input = widgets.Text(
    value='.',
    description='Workspace:',
    placeholder='Directory for file operations',
    style={'description_width': '120px'},
    layout=widgets.Layout(width='400px')
)

mock_toggle = widgets.Checkbox(
    value=False,
    description='Mock Mode (test without API)',
    indent=False,
    style={'description_width': 'auto'}
)

# Display configuration UI
config_box = widgets.VBox([
    model_dropdown,
    widgets.HTML(f"<p style='margin:5px 0;color:#888;'>Region: Sydney ({REGION})</p>"),
    temperature_dropdown,
    thinking_checkbox,
    thinking_budget_dropdown,
    workspace_input,
    max_turns_slider,
    mock_toggle,
], layout=widgets.Layout(padding='10px', border='1px solid #444', margin='10px 0', background='#2d2d2d'))

display(config_box)
display(HTML("<p style='color:#888;font-size:12px;'>Configure settings above, then run the next cell to start.</p>"))

In [ ]:
# ============================================================
# LAUNCH AGENT WITH CONFIGURATION
# ============================================================

from sagemaker_agent import CONFIG, create_chat_ui
from IPython.display import display, HTML

# Security settings
CONFIG.aws_bedrock_only = True          # Block ALL AWS services except Bedrock
CONFIG.require_tool_approval = True     # Show Approve/Deny dialog before execution
CONFIG.session_cost_limit = 5.0         # Default $5 — adjustable via Budget slider in UI

# Apply configuration from widgets above
CONFIG.model_id = AVAILABLE_MODELS[model_dropdown.value]
CONFIG.region = REGION  # Sydney
CONFIG.workspace = workspace_input.value
CONFIG.max_turns = max_turns_slider.value
CONFIG.mock_mode = mock_toggle.value
CONFIG.temperature = TEMPERATURE_OPTIONS[temperature_dropdown.value]
CONFIG.thinking_enabled = thinking_checkbox.value
CONFIG.thinking_budget = THINKING_BUDGET_OPTIONS[thinking_budget_dropdown.value]

# Display current config
thinking_str = f"Thinking: On (budget: {CONFIG.thinking_budget})" if CONFIG.thinking_enabled else "Thinking: Off"
display(HTML(f"""
<div style="background:#1e3a1e;padding:10px;border-radius:5px;margin:10px 0;color:#d4d4d4;">
<b>Configuration Applied</b><br>
Model: {model_dropdown.value} | Region: Sydney | Temp: {CONFIG.temperature}<br>
{thinking_str} | Budget: ${CONFIG.session_cost_limit:.0f} (adjust via slider in UI)<br>
Use <code>/skills</code> to list available skills, <code>/skill use &lt;name&gt;</code> to activate
</div>
"""))

# Launch the chat interface
create_chat_ui()

---

## Skills — Usage Guide (V4.8.0)

### What Are Skills?

Skills are **text files** (`SKILL.md`) that get injected into the AI's system prompt when activated. They don't add new tools — they change **how the agent thinks and works**. Like giving someone a checklist before they start a job.

**V4.8.0 change:** Skills NO LONGER auto-trigger on keywords. You must explicitly activate them via `/command`. This prevents unwanted skill activation when you're just having a conversation.

### The 10 Available Skills

| Skill | Command | What it does | Example |
|-------|---------|-------------|---------|
| **verify** | `/verify` | Adversarial BREAK testing — tries to break your code | "I rewrote auth. `/verify`" |
| **simplify** | `/simplify` | 3 parallel agents check reuse/quality/efficiency | "I added 200 lines. `/simplify`" |
| **done** | `/done quick` | Chains simplify → verify → SHIP/FAIL verdict | "Feature complete. `/done quick`" |
| **review** | `/review` | Full PR-style code review + security checklist | "`/review` the changes in src/" |
| **security-review** | `/security-review` | 3-phase vulnerability assessment (OWASP, injection, auth) | "`/security-review` the API" |
| **batch** | `/batch` | Parallel workers for bulk multi-file changes | "`/batch` rename across 50 files" |
| **report** | "create a report" | Professional Word/PDF with embedded charts | "create a report on test results" |
| **design** | `/design` | 2-3 options with tradeoffs before coding | "`/design` how to implement caching?" |
| **clara-review** | `/clara-review` | ClaRA project 5-phase production readiness | "`/clara-review`" |
| **powerbi** (x2) | Manual | Power BI dashboard generation (.pbip) | See skills/powerbi-dashboard/ |

### Skill Commands

```
/skills                  ← list all available skills
/skill use <name>        ← activate a skill (stays on for ALL messages until cleared)
/skill clear             ← deactivate ALL active skills
/verify                  ← shortcut: auto-activates verify + runs it
/done [full|quick]       ← pre-ship gate: simplify → verify → SHIP verdict
/simplify                ← shortcut: auto-activates simplify + runs it
/security-review         ← shortcut: auto-activates security-review + runs it
```

### When to Use What

```
Just coding?             → No skill needed. Agent follows built-in quality rules.
Done coding?             → /verify (quick break test) or /done quick (full gate)
Want feedback?           → /review (structured review of specific files)
Security-sensitive?      → /security-review (OWASP, injection, auth bypass)
Big refactor?            → /batch (parallel workers across many files)
Need a document?         → "create a report on X" (report skill auto-activates)
Architecture decision?   → /design (options with tradeoffs before coding)
```

### Creating Your Own Skills

Create a folder + `SKILL.md` in the `skills/` directory:

```markdown
---
name: my-skill
description: What this skill does
triggers: /my-skill
auto_trigger: false
---

Instructions for the agent here...
```

Set `auto_trigger: false` to prevent keyword matching. The agent discovers skills automatically on startup.

---

## Quick Reference

| Action | How |
|--------|-----|
| **Send message** | Type in input box, press Send |
| **Stop agent** | Click Stop button |
| **Check cost** | `/cost` |
| **Activate skill** | `/skill use review` or `/review` |
| **Deactivate skills** | `/skill clear` |
| **List skills** | `/skills` |
| **Verify code** | `/verify` |
| **Revert file** | `/revert filename.py` or `/revert all --yes` |
| **Compact context** | Click Compact button (or auto at 80%) |
| **Save/Load session** | Save button / Session dropdown + Load |
| **Resize chat** | Drag bottom edge or use Chat Height slider |

## Quick Start Examples

| Task | What to type |
|------|-------------|
| Read a file | "Read app.py" |
| Find files | "Find all Python files in this project" |
| Fix a bug | "Read app.py, find the bug, and fix it" |
| Create chart | "Create a bar chart from sales.csv showing revenue by product" |
| Create report | "Create a Word report summarizing the data in results.csv" |
| Run command | "Run git status" |
| Download file | "wget https://example.com/data.csv" |
| Validate data | "Read merged.csv and check if the row count matches the sources" |
| Code review | `/review` then "review the changes in src/" |
| Verify code | `/verify` |
| Explain code | "Explain app.py — purpose, architecture, data flow" |

## 25+ Tools

| Category | Tools | Approval? |
|----------|-------|-----------|
| **File** | read_file, write_file, edit_file, glob, grep, list_dir | write/edit need approval |
| **Exec** | bash, python_exec | Both need approval |
| **Docs** | create_word, create_excel, create_chart, create_pdf, create_markdown, create_notebook | Need approval |
| **Intelligence** | view_image (vision), semantic_search (code search), web_fetch (URL fetch) | web_fetch needs approval |
| **Agents** | skill (load checklist), task (spawn sub-agent), ask_user (ask you a question) | task needs approval |
| **State** | todo_write, todo_read | Auto |

## Sub-Agents (7 types)

| Type | What it does | Tools | Max turns |
|------|-------------|-------|-----------|
| **build** | Full development — read, write, execute | All 25+ | 25 |
| **plan** | Architecture analysis (read-only) | 11 | 15 |
| **explore** | Fast file search, codebase navigation | 5 | 10 |
| **verify** | Adversarial testing — tries to BREAK the code | 7 | 15 |
| **review** | Security, quality, performance review | 6 | 10 |
| **general** | General coding tasks | 11 | 15 |
| **fork** | Inherits parent context (cache-optimized) | Parent's tools | Parent's limit |

## Context Management

| Feature | What it does |
|---------|-------------|
| **Microcompact (70%)** | Replace old tool results with markers — buys headroom |
| **Auto-compact (80%)** | LLM summarizes conversation if microcompact insufficient |
| **Post-compact restore** | Re-injects last 3 recently-read files + TODO list after compact |
| **Post-compact cleanup** | Clears FILE_CACHE markers so agent re-reads correctly (V4.8.0) |
| **PTL retry** | Prompt-too-long: trim oldest messages, retry up to 3 times |
| **Circuit breaker** | After 3 compact failures, falls back to manual |
| **Cache miss warning** | Shows cost spike alert after compaction breaks cache (V4.8.0) |

## Memory System (4 types)

| Type | What it stores | Example |
|------|---------------|---------|
| **user** | Your role, preferences, expertise | "Senior Python dev, prefers pytest" |
| **feedback** | Corrections and confirmations | "Don't mock the database in tests" |
| **project** | Ongoing work context, deadlines | "Merge freeze after Thursday" |
| **reference** | Pointers to external resources | "Bugs tracked in Linear project INGEST" |

Stored in `memory.md`. Auto-extracted at session end. Capped at 200 lines / 25KB.

## Version History

| Version | Key Changes |
|---------|-------------|
| **V4.8.0** | Resizable chat, prefer chat answers, CSV/Excel validation, wget/bash allowed, budget display-only, skills user-controlled, post-compact FILE_CACHE cleanup, per-turn reminders, cache miss warning |
| **V4.7.2** | Workspace path resolution fix, auto-commit checkpoint, /regression command |
| **V4.7.1** | TODO restoration after compact, design skill |
| **V4.6.1** | Workspace info injection, glob fallthrough, skill discovery |
| **V4.3.2** | Cache-breakage detection, verify agent, bash git safety |
| **V4.3.0** | Diminishing returns, cold-cache microcompact, cache indicator |
| **V4.2.0** | Tool result caps, FILE_UNCHANGED_STUB, parallel RO tools |
| **V4.1.0** | Cache boundary, catastrophic blocks, 4-type memory |
| **V4.0.0** | Base V4 architecture |